# 📂 Практикум 2. Чтение энергетических данных разных форматов и первичный контроль качества

В инженерной аналитике данные редко приходят в одном удобном виде. Чаще приходится работать сразу с несколькими форматами: табличными выгрузками, текстовыми журналами, электронными таблицами и компактными JSON-описаниями наборов данных.


## 🧭 Введение и описание практикума

На старте исследования важно не только открыть файл, но и быстро понять, насколько данные пригодны для дальнейшего анализа. В этом практикуме рассматривается, как единым приемом прочитать несколько форматов файлов, сопоставить их структуру и получить первый профиль качества данных без ручных обходных действий.


### 🎯 Цель практикума

Показать, как на реальных учебных примерах читать файлы форматов `CSV`, `XLSX`, `TXT` и `JSON`, а затем выполнять первичную проверку структуры таблиц, типов столбцов и пропусков.


### ✨ Практическая значимость

Для задач электроэнергетики неоднородность исходных данных является нормой: телеметрия может храниться в текстовом экспорте, результаты обследования приходят в `XLSX`, а метаданные о наборе данных удобнее держать в `JSON`. Поэтому навыки универсального чтения и первичной проверки форматов являются базовой частью аналитической работы.


### 🧩 Задачи практикума

1. Прочитать четыре формата файлов из локального репозитория.
2. Сопоставить структуру и ключевые поля учебных наборов данных.
3. Построить первый профиль качества для табличного набора.
4. Показать, как разные форматы вписываются в единый исследовательский контур.


## 📚 Используемые библиотеки и методы

| Компонент | Назначение |
| --- | --- |
| `pandas` | чтение `CSV`, `XLSX`, `TXT` и табличная обработка |
| `json` | работа с каталогом наборов данных |
| вспомогательные функции курса | компактный вывод таблиц и смысловых блоков |


## 🗂 Источники данных

| Файл | Формат | Назначение |
| --- | --- | --- |
| `grid_stability_fragment.csv` | `CSV` | фрагмент набора для классификации устойчивости сети |
| `combined_cycle_power_plant_fragment.xlsx` | `XLSX` | пример набора для регрессии мощности |
| `household_power_fragment.txt` | `TXT` | пример временного ряда нагрузки |
| `dataset_catalog.json` | `JSON` | каталог учебных наборов данных |


In [ ]:
from pathlib import Path
import sys

import pandas as pd

# Добавляем корень репозитория в путь импорта, чтобы ноутбук запускался
# одинаково и из корня проекта, и из папки notebooks.
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from src.project_paths import project_root, sample_data_dir
from src.display_utils import display_callout, display_frame, display_metric_table, display_stage_note


import json

# Формируем пути к примерам разных форматов.
formats_dir = sample_data_dir() / "formats"
csv_path = formats_dir / "grid_stability_fragment.csv"
xlsx_path = formats_dir / "combined_cycle_power_plant_fragment.xlsx"
txt_path = formats_dir / "household_power_fragment.txt"
json_path = formats_dir / "dataset_catalog.json"

display_stage_note(
    "Подготовка источников",
    "Сначала важно убедиться, что все демонстрационные файлы доступны локально и могут быть открыты без ручной замены путей.",
)

source_frame = pd.DataFrame(
    {
        "Файл": [csv_path.name, xlsx_path.name, txt_path.name, json_path.name],
        "Существует": [path.exists() for path in [csv_path, xlsx_path, txt_path, json_path]],
        "Путь": [str(path) for path in [csv_path, xlsx_path, txt_path, json_path]],
    }
)
display_frame(source_frame)


### 🔎 Что показывает стартовая таблица источников

На этом шаге подтверждается главное условие воспроизводимости: все примеры находятся внутри репозитория и будут доступны читателю сразу после клонирования. Это особенно важно для учебного курса, где блокнот не должен требовать скрытых локальных файлов.


In [ ]:
from pathlib import Path
import sys

import pandas as pd

# Добавляем корень репозитория в путь импорта, чтобы ноутбук запускался
# одинаково и из корня проекта, и из папки notebooks.
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from src.project_paths import project_root, sample_data_dir
from src.display_utils import display_callout, display_frame, display_metric_table, display_stage_note


import json

formats_dir = sample_data_dir() / "formats"

# Читаем каждый формат тем способом, который соответствует его структуре.
csv_frame = pd.read_csv(formats_dir / "grid_stability_fragment.csv")
xlsx_frame = pd.read_excel(formats_dir / "combined_cycle_power_plant_fragment.xlsx")
txt_frame = pd.read_csv(formats_dir / "household_power_fragment.txt", sep=";")
catalog = json.loads((formats_dir / "dataset_catalog.json").read_text(encoding="utf-8"))

# Собираем сравнительную таблицу по наборам данных.
overview = pd.DataFrame(
    [
        {"Набор": "Фрагмент устойчивости сети", "Формат": "CSV", "Строк": len(csv_frame), "Столбцов": csv_frame.shape[1]},
        {"Набор": "Фрагмент мощности электростанции", "Формат": "XLSX", "Строк": len(xlsx_frame), "Столбцов": xlsx_frame.shape[1]},
        {"Набор": "Фрагмент временного ряда нагрузки", "Формат": "TXT", "Строк": len(txt_frame), "Столбцов": txt_frame.shape[1]},
        {"Набор": "Каталог учебных данных", "Формат": "JSON", "Строк": len(catalog), "Столбцов": 4},
    ]
)

display_stage_note(
    "Сопоставление форматов",
    "После чтения файлов важно сравнить их структуру и увидеть, что разные форматы могут быть включены в единый аналитический процесс.",
)
display_frame(overview)


### 🗃 Как интерпретировать сравнение форматов

Таблица показывает, что различие форматов не мешает выстраивать единый рабочий контур. `CSV` и `XLSX` удобны для табличных измерений, `TXT` часто используется для экспортов журналов или временных рядов, а `JSON` служит компактным контейнером для метаданных. Для аналитика важно не противопоставлять форматы, а уметь быстро приводить их к согласованному виду.


In [ ]:
from pathlib import Path
import sys

import pandas as pd

# Добавляем корень репозитория в путь импорта, чтобы ноутбук запускался
# одинаково и из корня проекта, и из папки notebooks.
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from src.project_paths import project_root, sample_data_dir
from src.display_utils import display_callout, display_frame, display_metric_table, display_stage_note


formats_dir = sample_data_dir() / "formats"
csv_frame = pd.read_csv(formats_dir / "grid_stability_fragment.csv")

# Для паспорта данных фиксируем типы столбцов, пропуски и число уникальных значений.
profile = pd.DataFrame(
    {
        "Поле": csv_frame.columns,
        "Тип": [str(csv_frame[column].dtype) for column in csv_frame.columns],
        "Пропусков": [int(csv_frame[column].isna().sum()) for column in csv_frame.columns],
        "Уникальных значений": [int(csv_frame[column].nunique()) for column in csv_frame.columns],
    }
)

display_stage_note(
    "Паспорт набора данных",
    "Теперь можно не только открыть файл, но и описать его как объект анализа: какие поля имеются, какие типы данных используются и есть ли проблемы с пропусками.",
)
display_frame(profile)


### 📑 Почему паспорт набора данных полезен уже в начале

Даже короткий профиль сразу показывает, какие поля являются числовыми, где следует ожидать признаки для модели, а где находятся целевые переменные или категориальные маркеры. Если бы на этом этапе обнаружились пропуски или неожиданные типы, это стало бы сигналом к дополнительной предобработке до начала моделирования.


In [ ]:
from pathlib import Path
import sys

import pandas as pd

# Добавляем корень репозитория в путь импорта, чтобы ноутбук запускался
# одинаково и из корня проекта, и из папки notebooks.
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from src.project_paths import project_root, sample_data_dir
from src.display_utils import display_callout, display_frame, display_metric_table, display_stage_note


import json

formats_dir = sample_data_dir() / "formats"
txt_frame = pd.read_csv(formats_dir / "household_power_fragment.txt", sep=";")
catalog = json.loads((formats_dir / "dataset_catalog.json").read_text(encoding="utf-8"))

# Сопоставляем каталог метаданных с фактическим содержимым учебных файлов.
catalog_frame = pd.DataFrame(catalog)
txt_preview = txt_frame.head(5)

display_stage_note(
    "Метаданные и фактическое содержимое",
    "Важно видеть не только саму таблицу, но и сопровождающую информацию о тематике, формате и происхождении набора данных.",
)
display_frame(catalog_frame)
display_frame(txt_preview)


### 🧠 Что дает сочетание метаданных и таблиц

Когда аналитик одновременно видит и содержимое файла, и его краткое описание, путь от чтения данных к постановке задачи становится короче. Метаданные помогают удерживать контекст: какой набор относится к классификации, какой удобен для регрессии, а какой отражает временной процесс и требует другой схемы анализа.


In [ ]:
from pathlib import Path
import sys

import pandas as pd

# Добавляем корень репозитория в путь импорта, чтобы ноутбук запускался
# одинаково и из корня проекта, и из папки notebooks.
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from src.project_paths import project_root, sample_data_dir
from src.display_utils import display_callout, display_frame, display_metric_table, display_stage_note


display_callout(
    "Результат практикума",
    "Файлы форматов CSV, XLSX, TXT и JSON успешно прочитаны, а их структура сведена к единому академическому описанию.",
    tone="success",
)


## 📌 Практическое значение результата

Завершая этот практикум, читатель получает не просто набор команд чтения файлов, а более важное умение: рассматривать формат как часть исследовательского контура. Это позволяет быстрее переходить от разнородных источников к согласованному набору данных, пригодному для разведочного анализа, построения признаков и обучения моделей.


## ✅ Что особенно важно запомнить

Работа с разными форматами не сводится к набору функций `read_*`. Содержательная часть начинается сразу после чтения: нужно проверить структуру, типы полей, наличие пропусков и место набора данных в общей логике исследования. Именно эта последовательность делает чтение файлов аналитическим действием, а не технической формальностью.


## ❓ Вопросы для самостоятельной работы

1. Почему `JSON` удобно использовать для метаданных о наборе данных, а не для самих больших временных рядов?
2. Какие дополнительные проверки качества вы бы добавили к паспорту данных перед обучением модели?
3. Чем различается подход к чтению `TXT`-выгрузок и табличных файлов `CSV`/`XLSX`?


## 📚 Источники

- pandas Documentation
- openpyxl Documentation
- Открытые учебные наборы данных, включенные в репозиторий курса
